In [ ]:
import geopandas as gpd
import pandas as pd
import os
os.chdir('/Users/jamie/Documents/GitHub/nycha') 

steam = "cleaned_data/combined_steam_2024.csv"
steamdf = pd.read_csv(steam)
steamgdf = gpd.GeoDataFrame(steamdf, geometry=gpd.points_from_xy(steamdf.longitude_nycha, steamdf.latitude_nycha))
steamgdf = steamgdf.set_crs(epsg=4326)

electricity = "cleaned_data/combined_electricity_2024.csv"
electricitydf = pd.read_csv(electricity)
electricitygdf = gpd.GeoDataFrame(electricitydf, geometry=gpd.points_from_xy(electricitydf.longitude_nycha, electricitydf.latitude_nycha))
electricitygdf = electricitygdf.set_crs(epsg=4326)

gas = "cleaned_data/combined_heating_gas_2024.csv"
gasdf = pd.read_csv(gas)
gasgdf = gpd.GeoDataFrame(gasdf, geometry=gpd.points_from_xy(gasdf.longitude_nycha, gasdf.latitude_nycha))
gasgdf = gasgdf.set_crs(epsg=4326)  

for name, gdf in [("Steam", steamgdf), ("Electricity", electricitygdf), ("Gas", gasgdf)]:
    print(f"\n{'='*40}")
    print(f"{name} GDF — head()")
    print('='*40)
    print(gdf.head())

electricitygdf.columns.tolist()
gasgdf.columns.tolist()





FileNotFoundError: [Errno 2] No such file or directory: '/Users/jamie/Documents/GitHub/nycha'

In [ ]:


steamgdf['steam_consumption_per_unit'] = steamgdf['consumption_(mlbs)'] / steamgdf['unitsres']
electricitygdf['electricity_consumption_per_unit'] = electricitygdf['consumption_(kwh)'] / electricitygdf['unitsres']
gasgdf['gas_consumption_per_unit'] = gasgdf['consumption_(therms)'] / gasgdf['unitsres']

steamgdf = steamgdf.rename(columns={'current_charges': 'steam_current_charges'})
electricitygdf = electricitygdf.rename(columns={'current_charges': 'electricity_current_charges'})
gasgdf = gasgdf.rename(columns={'current_charges': 'gas_current_charges'})


In [ ]:
#changing the column name to be more specific about what the consumption per unit is referring to, since we will be merging these dataframes later and want to avoid confusion
def keep_columns(gdf, consumption_col, charges_col):
    cols = [
        'BBL', 'development', 'tds_#', 'building_#', 'borough',
        'latitude_nycha', 'longitude_nycha', 'unitsres', 'unitstotal',
        'geometry', charges_col, consumption_col
    ]
    return gdf[cols]

steamgdf = keep_columns(steamgdf, 'steam_consumption_per_unit', 'steam_current_charges')
electricitygdf = keep_columns(electricitygdf, 'electricity_consumption_per_unit', 'electricity_current_charges')
gasgdf = keep_columns(gasgdf, 'gas_consumption_per_unit', 'gas_current_charges')


steamgdf = steamgdf.rename(columns={'current_charges': 'steam_current_charges'})
electricitygdf = electricitygdf.rename(columns={'current_charges': 'electricity_current_charges'})
gasgdf = gasgdf.rename(columns={'current_charges': 'gas_current_charges'})

steamgdf = keep_columns(steamgdf, 'steam_consumption_per_unit', 'steam_current_charges')
electricitygdf = keep_columns(electricitygdf, 'electricity_consumption_per_unit', 'electricity_current_charges')
gasgdf = keep_columns(gasgdf, 'gas_consumption_per_unit', 'gas_current_charges')



In [ ]:
#checking the head of each gdf to make sure the columns are correct and the data looks good before merging
#steamgdf.head()
#electricitygdf.head()
#gasgdf.head()
steamgdf_agg = steamgdf.drop(columns='geometry').groupby('BBL', as_index=False).agg({
    'development': 'first',
    'tds_#': 'first',
    'borough': 'first',
    'latitude_nycha': 'first',
    'longitude_nycha': 'first',
    'unitsres': 'first',
    'unitstotal': 'first',
    'steam_current_charges': 'first',
    'steam_consumption_per_unit': 'first'
})

In [ ]:
steamgdf = steamgdf.groupby(['BBL', 'building_#'], as_index=False).agg({
    **{col: 'first' for col in steamgdf.columns if col not in ['BBL', 'building_#', 'steam_current_charges', 'steam_consumption_per_unit']},
    'steam_current_charges': 'sum',
    'steam_consumption_per_unit': 'sum'
})

electricitygdf = electricitygdf.groupby(['BBL', 'building_#'], as_index=False).agg({
    **{col: 'first' for col in electricitygdf.columns if col not in ['BBL', 'building_#', 'electricity_current_charges', 'electricity_consumption_per_unit']},
    'electricity_current_charges': 'sum',
    'electricity_consumption_per_unit': 'sum'
})

gasgdf = gasgdf.groupby(['BBL', 'building_#'], as_index=False).agg({
    **{col: 'first' for col in gasgdf.columns if col not in ['BBL', 'building_#', 'gas_current_charges', 'gas_consumption_per_unit']},
    'gas_current_charges': 'sum',
    'gas_consumption_per_unit': 'sum'
})

In [ ]:
shared_cols = ['BBL', 'development', 'tds_#', 'borough', 'latitude_nycha', 'longitude_nycha', 'unitsres', 'unitstotal', 'geometry']

steamgdf_agg = steamgdf.groupby('BBL', as_index=False).agg({
    **{col: 'first' for col in shared_cols if col not in ['BBL']},
    'steam_current_charges': 'first',
    'steam_consumption_per_unit': 'first'
})

electricitygdf_agg = electricitygdf.groupby('BBL', as_index=False).agg({
    'electricity_current_charges': 'first',
    'electricity_consumption_per_unit': 'first'
})

gasgdf_agg = gasgdf.groupby('BBL', as_index=False).agg({
    'gas_current_charges': 'first',
    'gas_consumption_per_unit': 'first'
})

combined = steamgdf_agg.merge(electricitygdf_agg, on='BBL', how='outer')
combined = combined.merge(gasgdf_agg, on='BBL', how='outer')

print(combined.shape)
combined.head()



(529, 15)


,BBL,development,tds_#,borough,latitude_nycha,longitude_nycha,unitsres,unitstotal,geometry,steam_current_charges,steam_consumption_per_unit,electricity_current_charges,electricity_consumption_per_unit,gas_current_charges,gas_consumption_per_unit
0,1001110100,SMITH,27,None,40.709494,-73.996611,NaN,"1,947",POINT (-73.99661 40.70949),0.0,0.0,24748772.79,0.000000,25477629.24,0.000000
1,1002450001,TWO BRIDGES URA (SITE 7),266,None,40.710931,-73.986289,250.0,252,POINT (-73.98629 40.71093),0.0,0.0,4660235.14,122540.896000,4191484.22,24797.708000
2,1002550001,RUTGERS,99,None,40.712120,-73.992041,721.0,724,POINT (-73.99204 40.71212),0.0,0.0,10126310.63,93896.155340,10862544.50,23425.973509
3,1002560001,LA GUARDIA,76,None,40.712328,-73.989513,610.0,611,POINT (-73.98951 40.71233),0.0,0.0,15742133.17,174992.736066,24658465.54,58773.855033
4,1002560014,LA GUARDIA ADDITION,152,None,40.711761,-73.988516,150.0,150,POINT (-73.98852 40.71176),0.0,0.0,1208208.60,50909.653333,1792194.49,18787.087000


In [ ]:
# check for each utility
#for utility in ['steam', 'electricity', 'gas']:
    #mask = combined[f'{utility}_current_charges'].notna() & combined[f'{utility}_consumption_per_unit'].isna()
    #print(f"{utility}: {mask.sum()} rows have charges but missing consumption")

print(electricitygdf[electricitygdf['BBL'] == 1001110100][['BBL', 'building_#', 'electricity_current_charges', 'electricity_consumption_per_unit']])

           BBL building_#  electricity_current_charges  \
0   1001110100          1                  24748772.79   
1   1001110100         10                  24748772.79   
2   1001110100         11                  24748772.79   
3   1001110100         12                  24748772.79   
4   1001110100          2                  24748772.79   
5   1001110100          3                  24748772.79   
6   1001110100          4                  24748772.79   
7   1001110100          5                  24748772.79   
8   1001110100          6                  24748772.79   
9   1001110100          7                  24748772.79   
10  1001110100          8                  24748772.79   
11  1001110100          9                  24748772.79   

    electricity_consumption_per_unit  
0                                0.0  
1                                0.0  
2                                0.0  
3                                0.0  
4                                0.0  
5          

In [ ]:
borough_map = {
    '1': 'Manhattan',
    '2': 'Bronx',
    '3': 'Brooklyn',
    '4': 'Queens',
    '5': 'Staten Island'
}

combined['borough'] = combined['BBL'].astype(str).str[0].map(borough_map)
combined.head()


,BBL,development,tds_#,borough,latitude_nycha,longitude_nycha,unitsres,unitstotal,geometry,steam_current_charges,steam_consumption_per_unit,electricity_current_charges,electricity_consumption_per_unit,gas_current_charges,gas_consumption_per_unit
0,1001110100,SMITH,27,Manhattan,40.709494,-73.996611,NaN,"1,947",POINT (-73.99661 40.70949),0.0,0.0,24748772.79,0.000000,25477629.24,0.000000
1,1002450001,TWO BRIDGES URA (SITE 7),266,Manhattan,40.710931,-73.986289,250.0,252,POINT (-73.98629 40.71093),0.0,0.0,4660235.14,122540.896000,4191484.22,24797.708000
2,1002550001,RUTGERS,99,Manhattan,40.712120,-73.992041,721.0,724,POINT (-73.99204 40.71212),0.0,0.0,10126310.63,93896.155340,10862544.50,23425.973509
3,1002560001,LA GUARDIA,76,Manhattan,40.712328,-73.989513,610.0,611,POINT (-73.98951 40.71233),0.0,0.0,15742133.17,174992.736066,24658465.54,58773.855033
4,1002560014,LA GUARDIA ADDITION,152,Manhattan,40.711761,-73.988516,150.0,150,POINT (-73.98852 40.71176),0.0,0.0,1208208.60,50909.653333,1792194.49,18787.087000


In [ ]:
print(electricitygdf[electricitygdf['development'] == 'SMITH'][['BBL', 'development', 'electricity_current_charges', 'electricity_consumption_per_unit']])
print(gasgdf[gasgdf['development'] == 'SMITH'][['BBL', 'development', 'gas_current_charges', 'gas_consumption_per_unit']])

           BBL development  electricity_current_charges  \
0   1001110100       SMITH                  24748772.79   
1   1001110100       SMITH                  24748772.79   
2   1001110100       SMITH                  24748772.79   
3   1001110100       SMITH                  24748772.79   
4   1001110100       SMITH                  24748772.79   
5   1001110100       SMITH                  24748772.79   
6   1001110100       SMITH                  24748772.79   
7   1001110100       SMITH                  24748772.79   
8   1001110100       SMITH                  24748772.79   
9   1001110100       SMITH                  24748772.79   
10  1001110100       SMITH                  24748772.79   
11  1001110100       SMITH                  24748772.79   

    electricity_consumption_per_unit  
0                                0.0  
1                                0.0  
2                                0.0  
3                                0.0  
4                                0.0 

In [ ]:
print("Shape:", combined.shape)
print("\nColumns:", combined.columns.tolist())
print("\nNull counts:\n", combined.isnull().sum())
print("\nSample:\n")
combined.head()

Shape: (529, 15)

Columns: ['BBL', 'development', 'tds_#', 'borough', 'latitude_nycha', 'longitude_nycha', 'unitsres', 'unitstotal', 'geometry', 'steam_current_charges', 'steam_consumption_per_unit', 'electricity_current_charges', 'electricity_consumption_per_unit', 'gas_current_charges', 'gas_consumption_per_unit']

Null counts:
 BBL                                  0
development                          0
tds_#                                0
borough                              0
latitude_nycha                       0
longitude_nycha                      0
unitsres                            37
unitstotal                           5
geometry                             0
steam_current_charges                0
steam_consumption_per_unit           0
electricity_current_charges          0
electricity_consumption_per_unit     0
gas_current_charges                  0
gas_consumption_per_unit             0
dtype: int64

Sample:



,BBL,development,tds_#,borough,latitude_nycha,longitude_nycha,unitsres,unitstotal,geometry,steam_current_charges,steam_consumption_per_unit,electricity_current_charges,electricity_consumption_per_unit,gas_current_charges,gas_consumption_per_unit
0,1001110100,SMITH,27,Manhattan,40.709494,-73.996611,NaN,"1,947",POINT (-73.99661 40.70949),0.0,0.0,24748772.79,0.000000,25477629.24,0.000000
1,1002450001,TWO BRIDGES URA (SITE 7),266,Manhattan,40.710931,-73.986289,250.0,252,POINT (-73.98629 40.71093),0.0,0.0,4660235.14,122540.896000,4191484.22,24797.708000
2,1002550001,RUTGERS,99,Manhattan,40.712120,-73.992041,721.0,724,POINT (-73.99204 40.71212),0.0,0.0,10126310.63,93896.155340,10862544.50,23425.973509
3,1002560001,LA GUARDIA,76,Manhattan,40.712328,-73.989513,610.0,611,POINT (-73.98951 40.71233),0.0,0.0,15742133.17,174992.736066,24658465.54,58773.855033
4,1002560014,LA GUARDIA ADDITION,152,Manhattan,40.711761,-73.988516,150.0,150,POINT (-73.98852 40.71176),0.0,0.0,1208208.60,50909.653333,1792194.49,18787.087000


In [ ]:
def recalculate_consumption_per_unit(gdf):
    gdf['unitsres'] = pd.to_numeric(gdf['unitsres'].astype(str).str.replace(',', ''), errors='coerce')
    gdf['unitstotal'] = pd.to_numeric(gdf['unitstotal'].astype(str).str.replace(',', ''), errors='coerce')
    units = gdf['unitsres'].fillna(gdf['unitstotal'])
    gdf['steam_consumption_per_unit'] = gdf['steam_current_charges'] / units
    gdf['electricity_consumption_per_unit'] = gdf['electricity_current_charges'] / units
    gdf['gas_consumption_per_unit'] = gdf['gas_current_charges'] / units
    return gdf

combined_utilities= recalculate_consumption_per_unit(combined)
combined_utilities.head()


,BBL,development,tds_#,borough,latitude_nycha,longitude_nycha,unitsres,unitstotal,geometry,steam_current_charges,steam_consumption_per_unit,electricity_current_charges,electricity_consumption_per_unit,gas_current_charges,gas_consumption_per_unit
0,1001110100,SMITH,27,Manhattan,40.709494,-73.996611,NaN,1947.0,POINT (-73.99661 40.70949),0.0,0.0,24748772.79,12711.234099,25477629.24,13085.582558
1,1002450001,TWO BRIDGES URA (SITE 7),266,Manhattan,40.710931,-73.986289,250.0,252.0,POINT (-73.98629 40.71093),0.0,0.0,4660235.14,18640.940560,4191484.22,16765.936880
2,1002550001,RUTGERS,99,Manhattan,40.712120,-73.992041,721.0,724.0,POINT (-73.99204 40.71212),0.0,0.0,10126310.63,14044.813634,10862544.50,15065.942441
3,1002560001,LA GUARDIA,76,Manhattan,40.712328,-73.989513,610.0,611.0,POINT (-73.98951 40.71233),0.0,0.0,15742133.17,25806.775689,24658465.54,40423.714000
4,1002560014,LA GUARDIA ADDITION,152,Manhattan,40.711761,-73.988516,150.0,150.0,POINT (-73.98852 40.71176),0.0,0.0,1208208.60,8054.724000,1792194.49,11947.963267


In [ ]:
combined_utilities= gpd.GeoDataFrame(combined_utilities, geometry='geometry', crs='EPSG:4326')
combined_utilities.to_file("cleaned_data/combined_utilities_2024.geojson", driver="GeoJSON")

combined.drop(columns='geometry').to_csv("cleaned_data/combined_utilities_2024.csv", index=False)